# Evaluation questions for architecture comparison

A change in Shape or Binding accuracy is informative only if the questions isolate the intended skill. We construct balanced Conjunction cases, count-controlled Binding pairs, and counterfactual edits whose correct answers follow from the scene oracle. The selection set is used to compare models; confirmation is reserved for the final test.

In [1]:
from pathlib import Path
import sys, json, random, math, hashlib, copy
from collections import Counter
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.medium_visual_world.world import COLORS, SHAPES, ANGLES, Obj, Scene, validate_scene
from src.medium_visual_world.generate import HOLDOUT, TRAIN_PAIRS, ALL_PAIRS, read_jsonl
from src.medium_visual_world.representation_probing import oracle

OUT_DIR = PROJECT_ROOT / "outputs/medium_world/experiments/architecture_protocol"
TRAIN_DIR = PROJECT_ROOT / "outputs/medium_world/dataset"
SPLIT_SEEDS = {"selection": 2026111401, "confirmation": 2026111602}
N_CONJ, N_BIND_PAIRS, N_SWAP_PAIRS = 1000, 500, 500
N_OPERATOR, N_RECOLOR, N_OOD = 750, 500, 1000
CANONICAL_SCENES = {row["scene_id"] for fn in ("train.jsonl","id_dev.jsonl","ood_dev.jsonl","id_test.jsonl","ood_test.jsonl")
                      for row in read_jsonl(TRAIN_DIR/fn)}
SCENE_FINGERPRINTS = set(CANONICAL_SCENES)
print("Source-world scene fingerprints:",len(CANONICAL_SCENES))

Source-world scene fingerprints: 6824


## 1. Construct controlled questions

Scenes follow the original rendering and non-overlap rules. For Conjunction, distractor objects cannot repeat the queried Color or Shape; this prevents attribute counts from distinguishing same-object and separate-object positives. Swaps preserve geometry and attribute totals while changing which attributes belong together.

In [2]:
def pick_target(rng):
    return rng.choice(TRAIN_PAIRS)

def construct_scene(rng,forced,allowed=TRAIN_PAIRS,n=None,clearance=None):
    n = rng.randint(max(3,len(forced)),5) if n is None else n
    for attempt in range(2000):
        pairs = list(forced)+[rng.choice(allowed) for _ in range(n-len(forced))]
        objects=[]
        for i, (c,s) in enumerate(pairs):
            radius=rng.randint(5,8)
            angle=rng.choice(ANGLES[s])
            choices=[]
            for j in range(300):
                x=rng.randint(radius+3,64-radius-4)
                y=rng.randint(radius+3,64-radius-4)
                obj=Obj(c,s, radius,x,y,angle)
                gaps=[math.hypot(x-o.x,y-o.y)-radius-o.radius for o in objects]
                if any(g<=2 for g in gaps):continue
                if i==1 and clearance is not None:
                    gap=gaps[0]
                    if clearance=="close" and not (2<gap<=8):continue
                    if clearance=="far" and gap<=16:continue
                choices.append(obj)
                break
            if not choices:break
            objects.append(choices[0])
        if len(objects)==n:
            scene=Scene(tuple(objects))
            validate_scene(scene)
            if scene.scene_id() not in SCENE_FINGERPRINTS:
                return scene
    raise RuntimeError("No scene satisfies requested constraint")

def row(scene,task,color,shape,case,group,state=None,pair_id=None,extra=None):
    d=scene.as_dict()
    label=oracle(d,task,color,shape)
    sid=scene.scene_id()
    unique=f"{group}|{case}|{sid}|{task}|{color}|{shape}|{state}"
    rec=dict(example_id=unique,scene_id=sid,scene_key=sid,task=task,color=color,shape=shape,
             label=label,case=case,group=group,scene=d)
    if state is not None:rec["state"]=state
    if pair_id is not None:rec["pair_id"]=pair_id
    if extra:
        rec.update(extra)
    return rec

def mutate_color_swap(scene,i=0,j=1):
    d=scene.as_dict()
    d["objects"][i]["color"],d["objects"][j]["color"] = (d["objects"][j]["color"],d["objects"][i]["color"])
    from src.medium_visual_world.world import from_dict
    result=from_dict(d)
    validate_scene(result)
    assert Counter(o["color"] for o in d["objects"])==Counter(o.color for o in scene.objects)
    assert Counter(o["shape"] for o in d["objects"])==Counter(o.shape for o in scene.objects)
    return result

def clearance_between(scene,i=0,j=1):
    a,b = scene.objects[i],scene.objects[j]
    return math.hypot(a.x-b.x, a.y-b.y) - a.radius-b.radius

def sample_swap(rng,near="close",id_safe=True):
    for _ in range(1000):
        c,s = pick_target(rng)
        options=[(a,b) for a,b in TRAIN_PAIRS if a!=c and b!=s and
                 (((a,s) in TRAIN_PAIRS and (c,b) in TRAIN_PAIRS) if id_safe else
                  ((a,s) in HOLDOUT or (c,b) in HOLDOUT))]
        if not options:
            continue
        a,b = rng.choice(options)
        extra=[pair for pair in TRAIN_PAIRS if pair[0]!=c and pair[1]!=s and pair!=(a,b)]
        scene=construct_scene(rng,[(c,s),(a,b)],allowed=extra,clearance=near)
        swapped=mutate_color_swap(scene)
        if oracle(scene.as_dict(),"pair",c,s)==1 and oracle(swapped.as_dict(),"pair",c,s)==0:
            created=any((o.color,o.shape) in HOLDOUT for o in swapped.objects)
            assert bool(created)==(not id_safe)
            if swapped.scene_id() in SCENE_FINGERPRINTS:continue
            return scene,swapped,c,s,created,clearance_between(scene)
    raise RuntimeError("Cannot sample requested swap")

def sample_conjunction(rng,kind):
    for _ in range(1000):
        c,s = pick_target(rng)
        # Fillers carry neither queried attribute, so every stratum has exactly one queried-colour
        # object and/or one queried-shape object (count-controlled, like the Binding set).
        allowed=[p for p in TRAIN_PAIRS if p[0]!=c and p[1]!=s]
        if kind=="bound_positive":
            forced=[(c,s)]
        elif kind=="separate_positive":
            options=[(a,b) for a,b in TRAIN_PAIRS if a!=c and b==s]
            first=[(a,b) for a,b in TRAIN_PAIRS if a==c and b!=s]
            forced=[rng.choice(first),rng.choice(options)]
        elif kind=="shape_missing":
            forced=[rng.choice([p for p in TRAIN_PAIRS if p[0]==c and p[1]!=s])]
        elif kind=="color_missing":
            forced=[rng.choice([p for p in TRAIN_PAIRS if p[0]!=c and p[1]==s])]
        else:raise ValueError(kind)
        scene=construct_scene(rng,forced,allowed=allowed)
        truth=oracle(scene.as_dict(),"conjunction",c,s)
        actual_bound=oracle(scene.as_dict(),"pair",c,s)
        if kind=="bound_positive" and truth==actual_bound==1:return scene,c,s
        if kind=="separate_positive" and truth==1 and actual_bound==0:return scene,c,s
        if kind in ("color_missing","shape_missing") and truth==0:return scene,c,s
    raise RuntimeError(f"Cannot generate conjunction stratum {kind}")

In [3]:
def make_split(name,seed):
    rng=random.Random(seed)
    out={k:[] for k in ("conjunction","binding","swaps","operator","recolor","ood")}
    for kind in ("bound_positive","separate_positive","shape_missing","color_missing"):
        for _ in range(N_CONJ//4):
            scene,c,s=sample_conjunction(rng,kind)
            out["conjunction"].append(row(scene,"conjunction",c,s,kind,"conjunction"))
    # Positive/negative pairs differ ONLY by exchanging two object colors.
    for clearance in ("close","far"):
        for i in range(N_BIND_PAIRS//2):
            before,after,c,s,created,gap=sample_swap(rng,clearance,id_safe=True)
            group=f"{name}|binding|{clearance}|{i}"
            for state,scene in (("positive",before),("negative",after)):
                out["binding"].append(row(scene,"pair",c,s,clearance,"binding",state,group,
                                          dict(clearance=gap,adjacency=clearance,heldout_created=0)))
    # Separate swaps: retain both ID-safe and heldout-creating transformations.
    for i in range(N_SWAP_PAIRS):
        near="close" if i%2==0 else "far"
        safe = i%5!=0
        before,after, c,s, created,gap = sample_swap(rng,near,id_safe=safe)
        group=f"{name}|swap|{i}"
        for state,scene in (("before",before),("after",after)):
            for task in ("color","shape","conjunction","pair"):
                out["swaps"].append(row(scene,task,c,s,near,"swaps",state,group,
                                      dict(clearance=gap,adjacency=near,heldout_created=int(created))))
    # Three query/operator relation types, with no majority-answer shortcut.
    spec=[("discordant","separate_positive"),("positive","bound_positive"),
          ("negative","shape_missing"),("negative","color_missing")]
    ncounts={"discordant":250,"positive":250,"negative":250}
    for category,kind in spec:
        n=ncounts[category]//2 if category=="negative" else ncounts[category]
        for i in range(n):
            scene, c,s = sample_conjunction(rng,kind)
            group=f"{name}|operator|{category}|{kind}|{i}"
            for task in ("conjunction","pair"):
                out["operator"].append(row(scene,task,c,s,category,"operator",pair_id=group))
    # Identical geometry and Shape truth; change only an object's Color.
    for i in range(N_RECOLOR):
        present=i%2==0
        c,s = pick_target(rng)
        allowed=[p for p in TRAIN_PAIRS if p[1]!=s] if not present else TRAIN_PAIRS
        for attempt in range(100):
            forced=[(c,s)] if present else [rng.choice(allowed)]
            scene=construct_scene(rng,forced,allowed)
            old=scene.objects[0]
            new_colors=[c2 for c2 in COLORS if c2!=old.color and (c2,old.shape) in TRAIN_PAIRS]
            recolored=scene.as_dict()
            recolored["objects"][0]["color"]=rng.choice(new_colors)
            from src.medium_visual_world.world import from_dict
            changed=from_dict(recolored)
            validate_scene(changed)
            if changed.scene_id() not in SCENE_FINGERPRINTS:break
        else:raise RuntimeError("Unable to sample scene-disjoint recolouring")
        group=f"{name}|recolor|{i}"
        case="present" if present else "absent"
        for state,ss in (("before",scene),("after",changed)):
            out["recolor"].append(row(ss,"shape", c,s,case,"recolor",state,group,
                                      dict(radius=old.radius,angle=old.angle)))
    # Repaired OOD: paired three-object color cycle. Target pair appears only in the
    # positive endpoint; a DIFFERENT heldout pair appears in the negative endpoint.
    # Both images contain exactly one heldout object and have identical object counts,
    # color/shape multisets, positions, shapes and sizes. Only the binding changes.
    heldouts=sorted(HOLDOUT)
    for i in range(N_OOD//2):
        c,s = heldouts[i%len(heldouts)]
        cycles=[(a,b,x,d) for a,b in heldouts if a!=c and b!=s
                for x in COLORS for d in SHAPES if len({a,c,x})==3
                and (c,d) in TRAIN_PAIRS and (x,s) in TRAIN_PAIRS
                and (x,b) in TRAIN_PAIRS and (a,d) in TRAIN_PAIRS]
        a,b, x,d = rng.choice(cycles)
        before=construct_scene(rng,[(a,b),(c,d),(x,s)],allowed=TRAIN_PAIRS)
        transformed=before.as_dict()
        transformed["objects"][0]["color"]=x
        transformed["objects"][1]["color"]=a
        transformed["objects"][2]["color"]=c
        from src.medium_visual_world.world import from_dict
        after=from_dict(transformed)
        validate_scene(after)
        if after.scene_id() in SCENE_FINGERPRINTS:continue
        assert oracle(before.as_dict(),"pair",c,s)==0 and oracle(after.as_dict(),"pair",c,s)==1
        assert sum((o.color,o.shape) in HOLDOUT for o in before.objects)==1
        assert sum((o.color,o.shape) in HOLDOUT for o in after.objects)==1
        group=f"{name}|ood|{i}"
        for state,ss in (("before",before),("after",after)):
            out["ood"].append(row(ss,"pair", c,s,"yes" if state=="after" else "no","ood",
                                  state, group, extra={"any_heldout":1,"target_heldout":1}))
    for kind,rows in out.items():
        rng.shuffle(rows)
    return out

def write_jsonl(path, rows):
    with path.open("w",encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r,sort_keys=True, separators=(",",":"))+"\n")

SCENE_FINGERPRINTS = set(CANONICAL_SCENES)  # regenerate the same selection scenes when this cell is rerun
manifest={"design":"architecture_protocol_v1_count_controlled_v2","seed_map":SPLIT_SEEDS,"sets":{},"training_manifests":{}}
for fn in ("train.jsonl","id_dev.jsonl","ood_dev.jsonl"):
    manifest["training_manifests"][fn]=hashlib.sha256((TRAIN_DIR/fn).read_bytes()).hexdigest()
for name,seed in SPLIT_SEEDS.items():
    out_dir=OUT_DIR/name
    out_dir.mkdir(parents=True,exist_ok=True)
    sets=make_split(name,seed)
    manifest["sets"][name]={}
    for kind,records in sets.items():
        path=out_dir/(kind+".jsonl")
        write_jsonl(path,records)
        manifest["sets"][name][kind]={"rows":len(records),"scenes":len(set(r["scene_id"] for r in records)),
                                      "sha256":hashlib.sha256(path.read_bytes()).hexdigest()}
    scene_ids={r["scene_id"] for rows in sets.values() for r in rows}
    assert not (scene_ids & SCENE_FINGERPRINTS)
    SCENE_FINGERPRINTS.update(scene_ids)
    print("Generated:",name,{k:v["rows"] for k,v in manifest["sets"][name].items()})
(OUT_DIR/"selection_and_confirmation_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
display(pd.DataFrame([{ "split":s,"family":k,**v} for s,a in manifest["sets"].items() for k,v in a.items()]))

Generated: selection {'conjunction': 1000, 'binding': 1000, 'swaps': 4000, 'operator': 1500, 'recolor': 1000, 'ood': 1000}
Generated: confirmation {'conjunction': 1000, 'binding': 1000, 'swaps': 4000, 'operator': 1500, 'recolor': 1000, 'ood': 1000}


,split,family,rows,scenes,sha256
0,selection,conjunction,1000,1000,17fea5734de5bcc4b04cb8a71036e689de5107505d89d0...
1,selection,binding,1000,1000,6c87537477b09632ae48009b8908af2203fe0880c2379b...
2,selection,swaps,4000,1000,cf3ed9c5574b5543f35f26a7d653b081f0767e32eade60...
3,selection,operator,1500,750,28f665e39cbc90fb46f1c6cd94c3a71e7ba33cea019a4f...
4,selection,recolor,1000,1000,8481567c39781bdfe7d3bc31fa0570a5eb2ed0ce75cc83...
5,selection,ood,1000,1000,900910dd16d3ba949eaa1380d767a4d9102f0842ce5bb6...
6,confirmation,conjunction,1000,1000,7ea82fd401ef2625b8968737fa4913490e3e7f216f3609...
7,confirmation,binding,1000,1000,9eb144e8d74a430e70c6fa334a614d88aedd78a7f5d76c...
8,confirmation,swaps,4000,1000,dcfa331371e273b1ffe3fa49ae945e8eca969aa329f541...
9,confirmation,operator,1500,750,ac3f0932c8935f9f4665028201b494ac4cca2821089f0a...


## 2. Check for shortcuts

A balanced answer distribution does not prevent a model from exploiting scene statistics. We test whether nuisance metadata predicts labels, and separately whether queried-attribute counts distinguish Conjunction positives. Related questions and both endpoints of an edit remain in the same cross-validation fold.

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score

def metadata_features(r,family):
    objs=r["scene"]["objects"]
    # Nuisance-only for existence tasks: shape/color frequencies would themselves
    # give the oracle answer (that is NOT a shortcut audit).
    x=[len(objs),np.mean([o["radius"] for o in objs]),np.std([o["radius"] for o in objs])]
    x += [np.mean([o[d] for o in objs]) for d in ("x","y")]
    x += [COLORS.index(r["color"]),SHAPES.index(r["shape"])]
    if family=="recolor":
        x += [sum(o["color"]==c for o in objs) for c in COLORS]
    if family=="positives":  # queried counts do not reveal bound-vs-separate, so they are admissible here
        x += [sum(o["color"]==r["color"] for o in objs),sum(o["shape"]==r["shape"] for o in objs)]
    if family in ("binding","swaps","ood"):
        x += [sum(o["color"]==c for o in objs) for c in COLORS]
        x += [sum(o["shape"]==s for o in objs) for s in SHAPES]
        x += [sum(o["color"]==r["color"] for o in objs),sum(o["shape"]==r["shape"] for o in objs)]
    return x

def shortcut_audit(rows,family):
    x=np.asarray([metadata_features(r,family) for r in rows],dtype=np.float32)
    y=np.asarray([r["label"] for r in rows])
    groups=np.asarray([r.get("pair_id",r["scene_id"]) for r in rows])
    parts=[]
    for itr,ite in GroupKFold(5).split(x,y,groups):
        model=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=10,l2_regularization=10,random_state=27)
        model.fit(x[itr],y[itr])
        p=model.predict(x[ite])
        parts.append(balanced_accuracy_score(y[ite],p))
    return float(np.mean(parts)),float(np.std(parts,ddof=1))

audit=[]
for family in ("conjunction","binding","swaps","recolor","ood"):
    rows=list(read_jsonl(OUT_DIR/"selection"/(family+".jsonl")))
    if family=="swaps":rows=[r for r in rows if r["task"]=="pair"]
    score,sd=shortcut_audit(rows,family)
    audit.append(dict(family=family,cv_balanced_accuracy=score,cv_sd=sd,design_flag=score>0.60))
# The operator set has two different question labels per scene; audit its THREE
# scene-relation categories, not pooled Yes/No labels.
operator=list(read_jsonl(OUT_DIR/"selection/operator.jsonl"))
tuple_rows=list({r["pair_id"]:r for r in operator}.values())
x=np.asarray([metadata_features(r,"conjunction") for r in tuple_rows],dtype=np.float32)
y=np.asarray([r["case"] for r in tuple_rows])
groups=np.asarray([r["pair_id"] for r in tuple_rows])
op_scores=[]
for itr,ite in GroupKFold(5).split(x,y,groups):
    clf=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=10,l2_regularization=10,random_state=27)
    clf.fit(x[itr],y[itr])
    op_scores.append(balanced_accuracy_score(y[ite],clf.predict(x[ite])))
audit.append(dict(family="operator_three_case",cv_balanced_accuracy=float(np.mean(op_scores)),
                  cv_sd=float(np.std(op_scores,ddof=1)),design_flag=float(np.mean(op_scores))>0.45))
for family,source,cases in (("conjunction_bound_vs_separate","conjunction",("bound_positive","separate_positive")),
                            ("operator_positive_vs_discordant","operator",("positive","discordant"))):
    rows=[{**r,"label":int(r["case"]==cases[0])} for r in read_jsonl(OUT_DIR/"selection"/(source+".jsonl"))
          if r["case"] in cases and r["task"]=="conjunction"]
    score,sd=shortcut_audit(rows,"positives")
    audit.append(dict(family=family,cv_balanced_accuracy=score,cv_sd=sd,design_flag=score>0.60))
audit=pd.DataFrame(audit)
audit["note"]=np.where(audit.family.isin(("binding","swaps","ood")),
                       "paired by construction: nuisance features identical within pair, so 0.5 is exact","")
audit.to_csv(OUT_DIR/"selection/shortcut_audit.csv",index=False)
display(audit)
# Oracle single-cue rules are explicitly reported for Conjunction. These are
# task-semantic capabilities, not nuisance metadata; 75% is expected here.
conj=list(read_jsonl(OUT_DIR/"selection/conjunction.jsonl"))
rules=[]
for kind in ("color_only","shape_only","binding_like"):
    scores=[]
    for r in conj:
        objs=r["scene"]["objects"]
        c,s=r["color"],r["shape"]
        pred=(any(o["color"]==c for o in objs) if kind=="color_only" else
              any(o["shape"]==s for o in objs) if kind=="shape_only" else
              any(o["color"]==c and o["shape"]==s for o in objs))
        scores.append(int(pred==r["label"]))
    rules.append(dict(rule=kind,accuracy=float(np.mean(scores))))
display(pd.DataFrame(rules))
pd.DataFrame(rules).to_csv(OUT_DIR/"selection/conjunction_single_cue_baselines.csv",index=False)

,family,cv_balanced_accuracy,cv_sd,design_flag,note
0,conjunction,0.555421,0.030356,False,
1,binding,0.500000,0.000000,False,paired by construction: nuisance features iden...
2,swaps,0.500000,0.000000,False,paired by construction: nuisance features iden...
3,recolor,0.510000,0.036572,False,
4,ood,0.500000,0.000000,False,paired by construction: nuisance features iden...
5,operator_three_case,0.354667,0.011926,False,
6,conjunction_bound_vs_separate,0.507259,0.057489,False,
7,operator_positive_vs_discordant,0.514000,0.027019,False,


,rule,accuracy
0,color_only,0.75
1,shape_only,0.75
2,binding_like,0.75
